# CardioIA Fase 2 - Parte 1: Extração de Informações

## Sistema de Diagnóstico Automatizado com NLP

Este notebook implementa um sistema de extração de informações médicas que:
1. Carrega relatos de sintomas de pacientes
2. Utiliza um mapa de conhecimento (ontologia) para identificar sintomas
3. Sugere diagnósticos baseados nas associações sintoma-doença

**Dataset:** Combinação de dados do MedQuAD (NIH, CC BY 4.0) com dados sintéticos cardiológicos  
**Referência MedQuAD:** Ben Abacha & Demner-Fushman, BMC Bioinformatics, 2019  
**Autor:** CardioIA Team | **Versão:** 2.0

---
## 1. Imports e Configurações

In [1]:
import pandas as pd
import re
from collections import Counter

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 50)

print('✓ Bibliotecas importadas com sucesso!')

✓ Bibliotecas importadas com sucesso!


---
## 2. Carregamento de Dados

Utilizamos dois conjuntos de dados:
- **sintomas_pacientes.txt** — 10 frases de relatos de pacientes (obrigatório pela atividade)
- **mapa_conhecimento_expandido.csv** — ontologia expandida com ~500 associações sintoma→doença

> O mapa expandido combina dados do **MedQuAD/NIH** (traduzidos para português) com dados sintéticos cardiológicos.

In [2]:
def carregar_sintomas(caminho: str) -> list:
    """
    Carrega o arquivo de sintomas de pacientes.
    Retorna lista de frases não vazias.
    """
    try:
        with open(caminho, 'r', encoding='utf-8') as f:
            frases = [linha.strip() for linha in f if linha.strip()]
        if not frases:
            raise ValueError(f'Arquivo {caminho} está vazio.')
        return frases
    except FileNotFoundError:
        raise FileNotFoundError(f'Arquivo não encontrado: {caminho}')


def carregar_mapa_conhecimento(caminho: str) -> pd.DataFrame:
    """
    Carrega o mapa de conhecimento (ontologia sintoma-doença).
    Valida estrutura esperada: Sintoma1, Sintoma2, Doenca_Associada.
    """
    try:
        df = pd.read_csv(caminho, encoding='utf-8')
        colunas_esperadas = {'Sintoma1', 'Sintoma2', 'Doenca_Associada'}
        if not colunas_esperadas.issubset(set(df.columns)):
            raise ValueError(f'Colunas faltantes: {colunas_esperadas - set(df.columns)}')
        if len(df) == 0:
            raise ValueError(f'Arquivo {caminho} está vazio.')
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f'Arquivo não encontrado: {caminho}')


print('✓ Funções de carregamento definidas.')

✓ Funções de carregamento definidas.


In [3]:
# Caminhos dos arquivos
CAMINHO_SINTOMAS = '../data/sintomas_pacientes.txt'
CAMINHO_MAPA     = '../data/mapa_conhecimento_expandido.csv'

frases_pacientes  = carregar_sintomas(CAMINHO_SINTOMAS)
mapa_conhecimento = carregar_mapa_conhecimento(CAMINHO_MAPA)

print(f'✓ {len(frases_pacientes)} frases de pacientes carregadas')
print(f'✓ {len(mapa_conhecimento)} associações no mapa de conhecimento')

✓ 10 frases de pacientes carregadas
✓ 500 associações no mapa de conhecimento


In [4]:
print('=' * 80)
print('FRASES DE PACIENTES (10 relatos)')
print('=' * 80)
for i, frase in enumerate(frases_pacientes, 1):
    print(f'{i:2}. {frase}')

FRASES DE PACIENTES (10 relatos)
 1. Há dois dias estou com uma dor no peito que piora quando faço esforço físico.
 2. Sinto cansaço constante há uma semana, mesmo depois de descansar bem durante a noite.
 3. Tenho sentido falta de ar ao subir escadas desde o mês passado.
 4. Acordo no meio da noite com palpitações fortes que duram alguns minutos.
 5. Sinto tontura frequente quando me levanto rapidamente pela manhã.
 6. Há três dias tenho dor no peito acompanhada de falta de ar ao caminhar.
 7. Percebo um cansaço intenso ao fazer atividades simples como arrumar a casa.
 8. Tenho palpitações irregulares há duas semanas, principalmente após o almoço.
 9. Sinto uma pressão no peito quando fico nervoso ou estressado no trabalho.
10. Há alguns dias venho sentindo tontura e falta de ar ao mesmo tempo durante exercícios leves.


In [5]:
print('=' * 80)
print(f'MAPA DE CONHECIMENTO — primeiras 25 linhas (total: {len(mapa_conhecimento)})')
print('=' * 80)
display(mapa_conhecimento.head(25))

MAPA DE CONHECIMENTO — primeiras 25 linhas (total: 500)


,Sintoma1,Sintoma2,Doenca_Associada,Fonte
0,dor no peito,falta de ar,Infarto,Literatura Médica
1,dor no peito,sudorese,Infarto,Literatura Médica
2,dor no peito,náusea,Infarto,Literatura Médica
3,dor no peito,dor no braço esquerdo,Infarto,Literatura Médica
4,dor no peito,pressão no peito,Angina,Literatura Médica
5,pressão no peito,falta de ar,Angina,Literatura Médica
6,cansaço constante,inchaço nas pernas,Insuficiência Cardíaca,Literatura Médica
7,falta de ar,inchaço nos pés,Insuficiência Cardíaca,Literatura Médica
8,dispneia,fadiga,Insuficiência Cardíaca,Literatura Médica
9,palpitações,tontura,Arritmia,Literatura Médica


---
## 3. Pré-processamento de Texto

Normalização converte texto para minúsculas. A operação é **idempotente**: aplicar duas vezes produz o mesmo resultado.

In [6]:
def normalizar_texto(texto: str) -> str:
    """Normaliza texto para minúsculas (idempotente)."""
    if texto is None:
        return ''
    return texto.lower()


# Demonstração
exemplo = 'Estou com DOR NO PEITO e FALTA DE AR'
normalizado = normalizar_texto(exemplo)
print(f'Original:    "{exemplo}"')
print(f'Normalizado: "{normalizado}"')
print(f'Idempotente: {normalizado == normalizar_texto(normalizado)} ✓')

Original:    "Estou com DOR NO PEITO e FALTA DE AR"
Normalizado: "estou com dor no peito e falta de ar"
Idempotente: True ✓


---
## 4. Motor de Correspondência de Sintomas

In [7]:
def extrair_sintomas_conhecidos(mapa: pd.DataFrame) -> set:
    """Extrai todos os sintomas únicos do mapa de conhecimento."""
    sintomas = set(mapa['Sintoma1'].unique()) | set(mapa['Sintoma2'].unique())
    return sintomas


def identificar_sintomas(frase: str, sintomas_conhecidos: set) -> list:
    """
    Identifica sintomas presentes na frase do paciente.
    Usa busca por substring após normalização.
    """
    frase_norm = normalizar_texto(frase)
    return [s for s in sintomas_conhecidos if s in frase_norm]


sintomas_conhecidos = extrair_sintomas_conhecidos(mapa_conhecimento)
print(f'Total de sintomas no mapa: {len(sintomas_conhecidos)}')
print('Exemplos:', sorted(sintomas_conhecidos)[:10])

Total de sintomas no mapa: 32
Exemplos: ['aperto no peito', 'batimentos acelerados', 'batimentos irregulares', 'cansaço', 'cansaço constante', 'cansaço extremo', 'dificuldade para respirar', 'dispneia', 'dor de cabeça', 'dor na mandíbula']


---
## 5. Sugestor de Diagnósticos

In [8]:
def sugerir_diagnosticos(sintomas: list, mapa: pd.DataFrame) -> list:
    """
    Sugere diagnósticos baseados nos sintomas identificados.
    Retorna lista sem duplicatas.
    """
    if not sintomas:
        return []
    diagnosticos = set()
    for sintoma in sintomas:
        mask = (mapa['Sintoma1'] == sintoma) | (mapa['Sintoma2'] == sintoma)
        diagnosticos.update(mapa[mask]['Doenca_Associada'].tolist())
    return sorted(diagnosticos)


def processar_frase(frase: str, sintomas_conhecidos: set, mapa: pd.DataFrame) -> dict:
    """Pipeline completo: normalização → identificação → diagnóstico."""
    sintomas    = identificar_sintomas(frase, sintomas_conhecidos)
    diagnosticos = sugerir_diagnosticos(sintomas, mapa)
    return {
        'frase_original':       frase,
        'sintomas_identificados': sintomas,
        'diagnosticos_sugeridos': diagnosticos
    }


print('✓ Funções de diagnóstico definidas.')

✓ Funções de diagnóstico definidas.


---
## 6. Processamento das 10 Frases de Pacientes

In [9]:
def exibir_resultado(numero: int, resultado: dict):
    print(f"\n{'='*80}")
    print(f'PACIENTE {numero}')
    print(f"{'='*80}")
    print(f"\n📝 Relato:\n   \"{resultado['frase_original']}\"")

    print('\n🔍 Sintomas Identificados:')
    if resultado['sintomas_identificados']:
        for s in resultado['sintomas_identificados']:
            print(f'   • {s}')
    else:
        print('   ⚠️  Nenhum sintoma reconhecido.')

    print('\n💊 Diagnósticos Sugeridos:')
    if resultado['diagnosticos_sugeridos']:
        for d in resultado['diagnosticos_sugeridos']:
            print(f'   • {d}')
    else:
        print('   ⚠️  Sem diagnósticos sugeridos.')


resultados = []
for i, frase in enumerate(frases_pacientes, 1):
    r = processar_frase(frase, sintomas_conhecidos, mapa_conhecimento)
    resultados.append(r)
    exibir_resultado(i, r)


PACIENTE 1

📝 Relato:
   "Há dois dias estou com uma dor no peito que piora quando faço esforço físico."

🔍 Sintomas Identificados:
   • dor no peito

💊 Diagnósticos Sugeridos:
   • Angina
   • Arritmia
   • Cardiomiopatia
   • Doença Coronariana
   • Fibrilação Atrial
   • Hipertensão
   • Infarto
   • Insuficiência Cardíaca

PACIENTE 2

📝 Relato:
   "Sinto cansaço constante há uma semana, mesmo depois de descansar bem durante a noite."

🔍 Sintomas Identificados:
   • cansaço constante
   • cansaço

💊 Diagnósticos Sugeridos:
   • Angina
   • Arritmia
   • Cardiomiopatia
   • Doença Coronariana
   • Fibrilação Atrial
   • Hipertensão
   • Infarto
   • Insuficiência Cardíaca

PACIENTE 3

📝 Relato:
   "Tenho sentido falta de ar ao subir escadas desde o mês passado."

🔍 Sintomas Identificados:
   • falta de ar

💊 Diagnósticos Sugeridos:
   • Angina
   • Arritmia
   • Cardiomiopatia
   • Doença Coronariana
   • Fibrilação Atrial
   • Hipertensão
   • Infarto
   • Insuficiência Cardíaca
  

---
## 7. Resumo Estatístico

In [10]:
total          = len(resultados)
com_sintomas   = sum(1 for r in resultados if r['sintomas_identificados'])
sem_sintomas   = total - com_sintomas

todos_sintomas    = [s for r in resultados for s in r['sintomas_identificados']]
todos_diagnosticos = [d for r in resultados for d in r['diagnosticos_sugeridos']]

print('=' * 80)
print('RESUMO ESTATÍSTICO')
print('=' * 80)
print(f'\n📊 Frases processadas : {total}')
print(f'   Com sintomas       : {com_sintomas}')
print(f'   Sem sintomas       : {sem_sintomas}')

print('\n🔍 Sintomas mais frequentes:')
for sintoma, cnt in Counter(todos_sintomas).most_common():
    print(f'   • {sintoma}: {cnt}x')

print('\n💊 Diagnósticos mais sugeridos:')
for diag, cnt in Counter(todos_diagnosticos).most_common():
    print(f'   • {diag}: {cnt}x')

RESUMO ESTATÍSTICO

📊 Frases processadas : 10
   Com sintomas       : 10
   Sem sintomas       : 0

🔍 Sintomas mais frequentes:
   • falta de ar: 3x
   • dor no peito: 2x
   • cansaço: 2x
   • palpitações: 2x
   • tontura: 2x
   • cansaço constante: 1x
   • pressão no peito: 1x

💊 Diagnósticos mais sugeridos:
   • Angina: 10x
   • Arritmia: 10x
   • Cardiomiopatia: 10x
   • Doença Coronariana: 10x
   • Hipertensão: 10x
   • Infarto: 10x
   • Insuficiência Cardíaca: 10x
   • Fibrilação Atrial: 9x
   • Insuficiência Cardíaca Congestiva: 3x
   • Arritmia Ventricular: 2x
   • Bradicardia: 2x
   • Taquicardia: 2x


---
## 8. Conclusão

O sistema demonstrou como técnicas simples de NLP baseadas em regras podem ser usadas para:
- Identificar sintomas em relatos de pacientes
- Sugerir diagnósticos a partir de uma ontologia médica

### Fontes dos Dados
- **MedQuAD** (NIH) — 47.457 pares Q&A médicos, licença CC BY 4.0  
  Referência: Ben Abacha & Demner-Fushman, BMC Bioinformatics, 2019
- **Dados sintéticos CardioIA** — gerados com base em terminologia médica cardiológica

### Limitações
- Correspondência exata de strings (não captura variações linguísticas)
- Não considera contexto semântico
- Dados simulados — não representam pacientes reais

### Próximos Passos
- Parte 2: Classificador de risco com TF-IDF + Machine Learning